# ------------------------------------------------------------


In [1]:
import numpy as np
import pandas as pd

import holidays_preproc as hp
import Lags as lg
from extend_train import create_extend_train, load_favorita_tables
from handling_nan_values import fill_time_series_full_range, process_promotions_flexible, fill_oil_pchip
from JOIN_OIL_PROMOS import JOIN_OIL_PROMOS
from features_block import *
from feature_checks import check_features_quick
from train_utils import *

from types import SimpleNamespace
import time
from pathlib import Path
import joblib

# Detect available Parquet engine at import time
try:
    import pyarrow  # noqa: F401
    PARQUET_ENGINE = "pyarrow"
except Exception:
    try:
        import fastparquet  # noqa: F401
        PARQUET_ENGINE = "fastparquet"
    except Exception:
        PARQUET_ENGINE = None



In [2]:
# Configuration object
cfg = SimpleNamespace(
    paths=['train.csv', 'oil.csv', 'stores.csv', 'items.csv', 'holidays_events.csv'],
    cv=SimpleNamespace(
        stride_days=14,      # time-based CV stride (days)
        min_train_days=365   # minimal training length after embargo
    ),
    cols=SimpleNamespace(
        target="unit_sales",
        group_cols=["store_nbr", "item_nbr"]  # grouping keys for lag/rolling/expanding
    ),
    cache=SimpleNamespace(
        dir="cache",         # directory for cached fold-0 features/targets
        save_fold0=True
    ),
    model=SimpleNamespace(
        framework="xgboost",  # use the XGBoost backend
        params=dict(
            n_estimators=1200,
            learning_rate=0.10,
            max_depth=8,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            min_child_weight=1.0,
            enable_categorical=True,
            tree_method="hist",
            eval_metric="rmse",           # metric in constructor
            early_stopping_rounds=50,     # early stopping in constructor
            verbosity=1,                  # trainer logs
            random_state=42,
            n_jobs=-1
        )
    )
)

In [3]:
# 1) Load raw tables
print('1) Load raw tables')
dfs = load_favorita_tables(cfg.paths)
train = dfs['train']
train = train[:31000000] # sample - TO BE DELETED !
oil = dfs['oil']
stores = dfs['stores']
items = dfs['items']
holidays = dfs['holidays_events']

1) Load raw tables


In [4]:
# 2) Preprocess (your functions)
print('2) Preprocess')
train = create_extend_train(train, items, stores)    #KUBA   # City/State/FAMILY/Class #TODO
oil = fill_oil_pchip(oil)
oil = fill_time_series_full_range(oil)
promos = process_promotions_flexible(train, dataset_name="promos", strategy_for_missing_promo="nan")
holidays = hp.preprocess(holidays)

2) Preprocess
Input data: 1175 known prices, 1218 total dates
PCHIP interpolation applied successfully
   PCHIP input: 1218 known prices → 1704 complete dates

🔧 promos - strategy_for_missing_promo: 'nan'
   Original NaN values: 21,657,651
   Kept NaN values as separate category
   Final NaN count: 21,657,651
   Perfect for: XGBoost, LightGBM, CatBoost


In [5]:
# 3) Join auxiliary tables into train
print('3) Join auxiliary tables into train')

print('join holidays')
train = hp.merge_train(train, holidays, stores)

3) Join auxiliary tables into train
join holidays


In [6]:
print('join oil promos')
train = JOIN_OIL_PROMOS(train, oil, promos)

join oil promos


In [7]:
# 4) Build features for a given dataset slice
print('4) Build features for a given dataset slice')
# Moved to features_block.py

4) Build features for a given dataset slice


In [8]:
# 5) Time-series cross-validation windows
print('5) Time-series cross-validation windows')
# Tune window parameters so at least one CV window can be built
max_lag = 84        # was 365  # shorter max lag → smaller embargo
max_window = 14     # was 30   # smaller rolling window → smaller embargo
validation_windows = 3
size_of_validation_windows = 14
frozen_days = compute_frozen_days_from_max_lag(max_lag, max_window)          # often == max_lag

# [DEBUG] Check window feasibility before building windows
full_dates = unique_dates(train)
n_dates = len(full_dates)
val_size = size_of_validation_windows
embargo = frozen_days
min_train = cfg.cv.min_train_days

print('Old debug info:')
print(f"[DEBUG] n_dates={n_dates}, min_train_days={min_train}, embargo={embargo}, val_size={val_size}")
print(f"[DEBUG] required_min_dates={min_train + embargo + val_size}")
print(f"[DEBUG] feasibility={n_dates >= (min_train + embargo + val_size)}")

print('New debug info:')
print(f"[DEBUG] n_dates={len(full_dates)}, min_train_days={cfg.cv.min_train_days}, "
      f"embargo={compute_frozen_days_from_max_lag(max_lag, max_window)}, val_size={size_of_validation_windows}")
print(f"[DEBUG] required_min_dates={cfg.cv.min_train_days + compute_frozen_days_from_max_lag(max_lag, max_window) + size_of_validation_windows}")

windows = build_rolling_windows(    # TODO
                  full_dates = unique_dates(train),
                  n_windows = validation_windows,
                  val_size = size_of_validation_windows,
                  stride = cfg.cv.stride_days,
                  embargo = frozen_days,
                  min_train_days = cfg.cv.min_train_days)

fold_scores = []
models_per_fold = []

print('Built windows:')
print(windows)
print(f"Total windows: {len(windows)}")


5) Time-series cross-validation windows
Old debug info:
[DEBUG] n_dates=622, min_train_days=365, embargo=97, val_size=14
[DEBUG] required_min_dates=476
[DEBUG] feasibility=True
New debug info:
[DEBUG] n_dates=622, min_train_days=365, embargo=97, val_size=14
[DEBUG] required_min_dates=476
Built windows:
[((np.datetime64('2013-01-01T00:00:00.000000000'), np.datetime64('2014-05-27T00:00:00.000000000')), (np.datetime64('2014-09-02T00:00:00.000000000'), np.datetime64('2014-09-15T00:00:00.000000000'))), ((np.datetime64('2013-01-01T00:00:00.000000000'), np.datetime64('2014-05-13T00:00:00.000000000')), (np.datetime64('2014-08-19T00:00:00.000000000'), np.datetime64('2014-09-01T00:00:00.000000000'))), ((np.datetime64('2013-01-01T00:00:00.000000000'), np.datetime64('2014-04-29T00:00:00.000000000')), (np.datetime64('2014-08-05T00:00:00.000000000'), np.datetime64('2014-08-18T00:00:00.000000000')))]
Total windows: 3


In [9]:
# 6) Cross-validation loop
print('6) Cross-validation loop')
for fold_idx, (train_range, val_range) in enumerate(windows):
    tr = SLICE_BY_DATE(train, train_range)
    va = SLICE_BY_DATE(train, val_range)

    # Sort once (stable) by group_cols + date to avoid re-sorting inside feature functions
    t0 = time.perf_counter()
    tr = tr.sort_values(cfg.cols.group_cols + ['date'], kind='mergesort')
    va = va.sort_values(cfg.cols.group_cols + ['date'], kind='mergesort')
    print(f"[FEATS] Fold {fold_idx}: sorted tr={len(tr):,}, va={len(va):,} in {time.perf_counter()-t0:.2f}s")

    # Xtr = MAKE_FEATURES(tr, target_col=cfg.cols.target, group_cols=cfg.cols.group_cols,
    #                     pre_sorted=True, timers=True)
    # Xva = MAKE_FEATURES(va, target_col=cfg.cols.target, group_cols=cfg.cols.group_cols,
    #                     pre_sorted=True, timers=True)
    # Limit rolling/expanding for first comparison; pass group_cols from cfg; keep timers
    Xtr = MAKE_FEATURES(
        tr, target_col=cfg.cols.target, group_cols=cfg.cols.group_cols,
        lags=[7, 14],                 # keep only light lags
        windows=[7],                  # rolling 7 only
        rolling_stats=["mean"],       # mean only
        exp_stats=[],                 # disable expanding
        pre_sorted=True, timers=True
    )
    Xva = MAKE_FEATURES(
        va, target_col=cfg.cols.target, group_cols=cfg.cols.group_cols,
        lags=[7, 14],
        windows=[7],
        rolling_stats=["mean"],
        exp_stats=[],
        pre_sorted=True, timers=True
    )

    if fold_idx == 0:
        print("[INFO] Feature sanity check on fold 0 (train)")
        check_features_quick(Xtr, target=cfg.cols.target)

    y_tr = Xtr[cfg.cols.target];  Xtr = DROP_TARGET_COLS(Xtr, cfg.cols.target)
    y_va = Xva[cfg.cols.target];  Xva = DROP_TARGET_COLS(Xva, cfg.cols.target)

    # Save fold 0 features/targets to Parquet if engine is available, else fallback to pickle
    if fold_idx == 0 and getattr(cfg, "cache", None) is not None and getattr(cfg.cache, "save_fold0", True):
        cache_dir = Path(cfg.cache.dir)
        cache_dir.mkdir(parents=True, exist_ok=True)

        if PARQUET_ENGINE:
            # Use Parquet with the detected engine (optionally add compression="snappy"/"zstd")
            Xtr.to_parquet(cache_dir / "Xtr_fold0.parquet", index=False, engine=PARQUET_ENGINE)
            Xva.to_parquet(cache_dir / "Xva_fold0.parquet", index=False, engine=PARQUET_ENGINE)
            y_tr.to_frame(name=cfg.cols.target).to_parquet(cache_dir / "y_tr_fold0.parquet", engine=PARQUET_ENGINE)
            y_va.to_frame(name=cfg.cols.target).to_parquet(cache_dir / "y_va_fold0.parquet", engine=PARQUET_ENGINE)
            print(f"[CACHE] Saved fold 0 to {cache_dir} (engine={PARQUET_ENGINE})")
        else:
            # Fallback: pickle (larger files, but no extra deps)
            Xtr.to_pickle(cache_dir / "Xtr_fold0.pkl")
            Xva.to_pickle(cache_dir / "Xva_fold0.pkl")
            y_tr.to_frame(name=cfg.cols.target).to_pickle(cache_dir / "y_tr_fold0.pkl")
            y_va.to_frame(name=cfg.cols.target).to_pickle(cache_dir / "y_va_fold0.pkl")
            print(f"[CACHE] Parquet engine not available. Saved pickle files to {cache_dir}")

    model = TRAIN_MODEL(Xtr, y_tr, Xva, y_va, cfg.model)         # LightGBM / XGBoost + early stop
    preds = PREDICT(model, Xva, cfg.model)

    fold_scores.append( EVALUATE_METRICS(y_va, preds, metrics = ["RMSE","WMAPE"]) )
    models_per_fold.append(model)

    models_dir = Path("artifacts/models")
    models_dir.mkdir(parents=True, exist_ok=True)
    model_path = models_dir / f"{cfg.model.framework}_fold{fold_idx}.pkl"
    joblib.dump(model, model_path)
    print(f"[MODEL] Saved {cfg.model.framework} fold {fold_idx} to {model_path}")

cv_summary = AGGREGATE_FOLD_SCORES(fold_scores)
print("[CV] fold_scores:", fold_scores)
print("[CV] summary:", cv_summary)


6) Cross-validation loop
[FEATS] Fold 0: sorted tr=24,478,670, va=940,560 in 5.57s
[FEATS] date features in 7.03s
[FEATS] seasonality features in 28.76s
[FEATS] promo features in 109.70s
[FEATS] lag features in 3.92s
[FEATS] rolling features in 22.13s
[FEATS] expanding features in 3.84s
[FEATS] date features in 0.45s
[FEATS] seasonality features in 0.70s
[FEATS] promo features in 54.37s
[FEATS] lag features in 0.11s
[FEATS] rolling features in 14.80s
[FEATS] expanding features in 0.08s
[INFO] Feature sanity check on fold 0 (train)
[FEATS] shape=(24478670, 47), memory=10795.9 MB
[FEATS] dtype counts:
float32           7
int32             7
int64             5
float64           5
bool              5
object            5
UInt8             4
UInt32            2
Int8              2
datetime64[ns]    1
category          1
int8              1
boolean           1
Int16             1
[FEATS] top 15 NA ratios:
days_since_last_promo         0.997415
days_until_next_promo         0.934399
promo_in_

In [10]:
# HYPERTUNING

# 7) (Optional) Refit final model on all history up to the end of training
#    (optionally up to scoring_period.t_min - 1)
print('7) (Optional) Refit final model on all history up to the end of training')
full_hist = SLICE_BY_DATE(train, [MIN_DATE(train), MAX_DATE(train)])
X_full = MAKE_FEATURES(full_hist, max_lag)
y_full = X_full[cfg.cols.target];  X_full = DROP_TARGET_COLS(X_full, cfg.cols.target)

final_model = TRAIN_MODEL_NO_VALIDATION(X_full, y_full, cfg.model)

7) (Optional) Refit final model on all history up to the end of training


KeyboardInterrupt: 